In [1]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer

/media/dzielinski06/HDD1/AI894 - Capstone/Complete Collision Recorder/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Map src directory
import sys
import os
root_dir = os.path.abspath(os.path.join(os.getcwd(), "../."))
print("Root Directory: ", root_dir)
src_dir = os.path.join(root_dir,"src")
print("Src Directory: ", src_dir)
sys.path.append(src_dir)
data_dir = os.path.join(root_dir,"data")
print("Data Directory: ", data_dir)

Root Directory:  /media/dzielinski06/HDD1/AI894 - Capstone/Complete Collision Recorder
Src Directory:  /media/dzielinski06/HDD1/AI894 - Capstone/Complete Collision Recorder/src
Data Directory:  /media/dzielinski06/HDD1/AI894 - Capstone/Complete Collision Recorder/data


In [3]:
def concatenate_texts(row):
    cad_text = row['CAD_TEXT'] if pd.notna(row['CAD_TEXT']) else ""
    oh_text = row['OH1_TEXT'] if pd.notna(row['OH1_TEXT']) else ""
    
    if oh_text:  # If 'OH_TEXT' is not an empty string
        return cad_text + "POLICE NARRATIVE \n\n " + oh_text
    else:
        return cad_text  # If 'OH_TEXT' is empty, return only 'CAD_TEXT'

# Apply the function to the dataframe to create a new concatenated column
training_df = pd.read_csv(os.path.join(data_dir, "processed", "training_df.csv"))
training_df['concatenated_text'] = training_df.apply(concatenate_texts, axis=1)
training_df = training_df[training_df["BIKE_CLE_TEXT"].notnull() & training_df["concatenated_text"].notnull()]
training_df = training_df.dropna(subset=['concatenated_text', 'BIKE_CLE_TEXT'])
training_df = training_df[training_df['concatenated_text'].str.strip() != '']
training_df = training_df[training_df['BIKE_CLE_TEXT'].str.strip() != '']



In [4]:
from datasets import Dataset

dataset = Dataset.from_pandas(training_df[['concatenated_text', 'BIKE_CLE_TEXT']])
tokenizer = AutoTokenizer.from_pretrained('mistralai/Mistral-7B-v0.1')
tokenizer.pad_token_id = 0
tokenizer.padding_side = "left"

def tokenize_inputs_and_labels(examples):
    # Tokenize input (concatenated_text)
    inputs = tokenizer(examples['concatenated_text'], padding='max_length', truncation=True, max_length=2048)
    
    # Tokenize target labels (BIKE_CLE_TEXT)
    labels = tokenizer(examples['BIKE_CLE_TEXT'], padding='max_length', truncation=True, max_length=2048)
    
    # Use input_ids from both tokenizations
    inputs["labels"] = labels["input_ids"]
    
    # Replace padding token id (usually -100) so it's ignored in the loss calculation
    inputs["labels"] = [
        [(label if label != tokenizer.pad_token_id else -100) for label in labels_seq] 
        for labels_seq in inputs["labels"]
    ]
    
    return inputs

# Apply tokenization to the entire dataset
tokenized_dataset = dataset.map(tokenize_inputs_and_labels, batched=True)

train_test_split = tokenized_dataset.train_test_split(test_size=0.95)

# Separate the datasets
train_dataset = train_test_split['train']
validation_dataset = train_test_split['test']

Map: 100%|██████████| 1315/1315 [00:02<00:00, 582.25 examples/s]


In [5]:
from transformers import AutoModelForCausalLM
import torch

# Load the Mistral-7B model
model = AutoModelForCausalLM.from_pretrained('mistralai/Mistral-7B-v0.1')
device = torch.device("cpu")
model.to(device)


Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.92s/it]


MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): MistralRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
     

In [6]:
from peft import get_peft_model, LoraConfig, TaskType

# Set up LoRA configuration for CPU
config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, 
    lora_alpha=16, 
    lora_dropout=0.1, 
    target_modules=["q_proj", "v_proj"]  # Fine-tune specific layers
)

# Apply LoRA configuration to the model
peft_model = get_peft_model(model, config)
peft_model.to(device)  # Ensure LoRA-adapted model is on CPU

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralSdpaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Lin

In [7]:
from transformers import Trainer, TrainingArguments

# Set up training arguments for CPU-only execution
training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    per_device_train_batch_size=1,  # Keep the batch size small for CPU training
    num_train_epochs=3,
    logging_dir="./logs",
    logging_steps=10,
    save_steps=500,
    save_total_limit=2,
    no_cuda=True  # IMPORTANT: This ensures no GPU is used at all
)

# Initialize the Trainer for CPU
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
)

# Fine-tune the model on CPU
trainer.train()


/media/dzielinski06/HDD1/AI894 - Capstone/Complete Collision Recorder/.venv/lib/python3.12/site-packages/transformers/training_args.py:1560: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(
  5%|▌         | 10/195 [23:20<7:08:29, 138.97s/it]

{'loss': 15.9222, 'grad_norm': 19.872440338134766, 'learning_rate': 4.7435897435897435e-05, 'epoch': 0.15}


 10%|█         | 20/195 [46:08<6:45:06, 138.89s/it]

{'loss': 14.5875, 'grad_norm': 49.76502990722656, 'learning_rate': 4.4871794871794874e-05, 'epoch': 0.31}


 15%|█▌        | 30/195 [1:09:16<6:21:52, 138.86s/it]

...

 97%|█████████▋| 190/195 [7:09:57<11:49, 141.92s/it]

{'loss': 3.5659, 'grad_norm': 20.2254581451416, 'learning_rate': 1.282051282051282e-06, 'epoch': 2.92}


100%|██████████| 195/195 [7:21:36<00:00, 135.88s/it]

{'train_runtime': 26496.5237, 'train_samples_per_second': 0.007, 'train_steps_per_second': 0.007, 'train_loss': 6.111750861925956, 'epoch': 3.0}


TrainOutput(global_step=195, training_loss=6.111750861925956, metrics={'train_runtime': 26496.5237, 'train_samples_per_second': 0.007, 'train_steps_per_second': 0.007, 'total_flos': 1.704644510220288e+16, 'train_loss': 6.111750861925956, 'epoch': 3.0})

In [10]:
peft_model.save_pretrained("./fine_tuned_mistralai")
tokenizer.save_pretrained("./fine_tuned_mistralai")



('./fine_tuned_mistralai/tokenizer_config.json',
 './fine_tuned_mistralai/special_tokens_map.json',
 './fine_tuned_mistralai/tokenizer.json')

: 